# Tetris-Multiplayer-RL — Colab setup

Bootstrap a Colab Linux x86_64 environment to:

1. clone the repo
2. install build deps (cmake, g++, python-dev, pybind11, torch, numpy)
3. build the headless `tetris_py` pybind11 module **without raylib** (`-DTETRIS_BUILD_GAME=OFF -DTETRIS_BUILD_PY=ON`)
4. drop the resulting `.so` into `python/sim/`
5. smoke-test the import and check the cross-platform state-hash parity gate

After this notebook completes, you can `from sim import SimGame` and `from common.models import TetrisPolicyNet` from any other Colab notebook in this runtime. Exported ONNX policies use the same checkpoint format for the in-game bot.

## 1. Clone the repo

Replace `<USER>` with your GitHub user (or use your fork URL).

In [1]:
REPO_URL = 'https://github.com/Rein-ArXiv/Tetris-Multiplayer-RL.git'

import os
if not os.path.exists('Tetris-Multiplayer-RL'):
    !git clone $REPO_URL
%cd Tetris-Multiplayer-RL
!git pull

/content/Tetris-Multiplayer-RL
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (19/19), done.
remote: Total 49 (delta 19), reused 19 (delta 19), pack-reused 30 (from 1)
Unpacking objects: 100% (49/49), 45.36 KiB | 749.00 KiB/s, done.
From https://github.com/Rein-ArXiv/Tetris-Multiplayer-RL
   86260cf..a34a4cb  main       -> origin/main
Updating 86260cf..a34a4cb
Fast-forward
 .gitignore                                   |   6 +
 README.md                                    |  39 +-
 bindings/tetris_py.cpp                       |   3 +
 model/bots.cfg.example                       |  21 ++
 model/bots/README.md                         |  47 +++
 platform/platform.h                          |   2 +
 platform/sdl.cpp                             |   2 +
 pyproject.toml                               |   1 +
 python/netbot/export_onnx.py                 |  53 ++-
 python/netbot/framing.py                     |   3 +-
 python/requirements-colab.txt                |   1 +


## 2. Install build dependencies

Colab already has `cmake`, `g++`, and `python3-dev`, but pin them anyway so this notebook works on a cold runtime. Python packages come from `python/requirements-colab.txt` so the baseline PPO loop has Gymnasium available.

In [2]:
!apt-get install -qq cmake g++ python3-dev > /dev/null
!pip install -q -r python/requirements-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.4 MB/s eta 0:00:00


## 3. Build the pybind11 module (no raylib)

`-DTETRIS_BUILD_GAME=OFF` skips the raylib executable so we don't need to apt-install raylib at all. `-DTETRIS_BUILD_PY=ON` enables the `tetris_py` target. We pass pybind11's CMake dir explicitly so the find_package picks up the pip-installed copy.

In [3]:
import subprocess, shutil, glob

PYBIND11_DIR = subprocess.check_output(['python', '-m', 'pybind11', '--cmakedir']).decode().strip()
print('pybind11 cmake dir:', PYBIND11_DIR)

!mkdir -p build
!cd build && cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DTETRIS_BUILD_GAME=OFF \
    -DTETRIS_BUILD_PY=ON \
    -DTETRIS_BUILD_TEST=ON \
    -Dpybind11_DIR=$PYBIND11_DIR
!cmake --build build -j --target tetris_py sim_hash_dump

# Drop the freshly built .so next to python/sim/__init__.py so `from sim import SimGame` finds it.
for so in glob.glob('build/tetris_py*.so'):
    print('copying', so, '-> python/sim/')
    shutil.copy(so, 'python/sim/')

pybind11 cmake dir: /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11
-- OpenSSL found: HTTPS meta client enabled
-- Configuring done (0.2s)
-- Generating done (0.0s)
-- Build files have been written to: /content/Tetris-Multiplayer-RL/build
[ 25%] Building CXX object CMakeFiles/tetris_py.dir/bindings/tetris_py.cpp.o
[ 50%] Linking CXX shared module tetris_py.cpython-312-x86_64-linux-gnu.so
[100%] Built target tetris_py
[100%] Built target sim_hash_dump
copying build/tetris_py.cpython-312-x86_64-linux-gnu.so -> python/sim/


## 4. Sanity import

Adds `python/` to `sys.path` and imports the wrapper. Prints the initial state hash for a known seed — this is the value that should match the Windows local build (cross-platform determinism gate).

In [4]:
import sys
sys.path.insert(0, 'python')

from sim import SimGame, Placement
g = SimGame(42)
print('initial state_hash:', hex(g.state_hash()))
print('current piece id  :', g.current_block_id())
print('next piece id     :', g.next_block_id())
print('legal placements  :', len(g.legal_placements()))
for p in g.legal_placements()[:5]:
    print('   ', p)

initial state_hash: 0x53d0178b1e0cb208
current piece id  : 6
next piece id     : 1
legal placements  : 33
    Placement(col=0, rot=0)
    Placement(col=1, rot=0)
    Placement(col=2, rot=0)
    Placement(col=3, rot=0)
    Placement(col=4, rot=0)


## 5. Run the C++ determinism dump

`build/sim_hash_dump` is the C++ test driver. Capture its stdout and write it to `python/tests/_sim_hash_dump.txt` — that's the file `test_determinism_crossplatform.py` parses to compare against the Python replay.

Repeat this exact step on your Windows machine and `diff` the two files. They MUST be byte-identical; if they're not, you have a cross-platform determinism bug to fix before any RL training is meaningful.

In [5]:
!build/sim_hash_dump > python/tests/_sim_hash_dump.txt
!head -40 python/tests/_sim_hash_dump.txt

==== seed 0x0000000000000001 ====
seed=0x0000000000000001
initial_hash=0xfb7249998a4f8ed6
step=000 mask=0x00 ticks=30 total_ticks=30 score=0 over=0 hash=0x2029ed42eb3493d7
step=001 mask=0x01 ticks=1 total_ticks=31 score=0 over=0 hash=0x9c0bc617492ab83f
step=002 mask=0x01 ticks=1 total_ticks=32 score=0 over=0 hash=0x9eeafcf1e6f3dc43
step=003 mask=0x01 ticks=1 total_ticks=33 score=0 over=0 hash=0x9c0616478f23c663
step=004 mask=0x08 ticks=1 total_ticks=34 score=0 over=0 hash=0x05cacf3cfe9d2f9c
step=005 mask=0x10 ticks=2 total_ticks=36 score=0 over=0 hash=0xf372c4d951746c18
step=006 mask=0x00 ticks=5 total_ticks=41 score=0 over=0 hash=0xac65bc277c3f4d1d
step=007 mask=0x02 ticks=1 total_ticks=42 score=0 over=0 hash=0x1f2136e103c04a5d
step=008 mask=0x02 ticks=1 total_ticks=43 score=0 over=0 hash=0x52502d2a8195bb3d
step=009 mask=0x08 ticks=1 total_ticks=44 score=0 over=0 hash=0x742f098262abf186
step=010 mask=0x08 ticks=1 total_ticks=45 score=0 over=0 hash=0x6ccb9261d3a5721c
step=011 mask=0x04

## 6. Verify the common/ shared layer imports

If this cell runs without error then the architecture / observation / checkpoint code is wired up and a Colab-trained checkpoint can be exported for the in-game ONNX bot.

In [6]:
import torch
from common.models import TetrisPolicyNet
from common.obs import build_observation
from common.action_mask import legal_mask
from common.checkpoint import save_checkpoint, load_checkpoint

model = TetrisPolicyNet()
obs = build_observation(SimGame(42))
batched = {k: v.unsqueeze(0) for k, v in obs.items()}
logits, value = model(**batched)
print('policy logits shape:', tuple(logits.shape))
print('value shape       :', tuple(value.shape))

save_checkpoint(model, '/tmp/toy.pt', extra={'training_steps': 0})
loaded = load_checkpoint('/tmp/toy.pt')
print('checkpoint round-trip OK')

policy logits shape: (1, 40)
value shape       : (1,)
checkpoint round-trip OK
